Evaluating ASR

In [13]:
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec spacy
!python -m spacy download en_core_web_trf

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 65.6 MB/s  0:00:06:00:0100:01
  Using cached spacy_curated_transformers-0.3.0-py2.py3-none-any.whl.metadata (2.7 kB)
  Using cached curated_transformers-0.1.1-py2.py3-none-any.whl.metadata (965 bytes)
  Using cached curated_tokenizers-0.0.10-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (2.0 kB)
Using cached spacy_curated_transformers-0.3.0-py2.py3-none-any.whl (236 kB)
Using cached curated_tokenizers-0.0.10-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.3 MB)
Using cached curated_transformers-0.1.1-py2.py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [en-core-web-trf] [en-core-web-trf]
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


## Setup

In [2]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto")
print(f"Model loaded on {model.device} with dtype {model.dtype}")

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


## Test Data Setup

In [3]:
import torchaudio
from IPython.display import Audio, display




dataset_paths = [("/training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test", "voxpopuli"),
                 ("/training-1/asr/kaldi_data/cv19/en/test", "commonvoice"),
                 ("/training-1/asr/kaldi_data/singleword/en/test", "commonvoice_singlewords"),
                 ("/training-1/asr/kaldi_data/mls/en/test", "librispeech")

                 
                ]# other datasets
# Mozilla Commonvoice: /training-1/asr/kaldi_data/cv19/en/test /de/ /fr/ /es/
# Mozilla Commonvoice Single Words: /training-1/asr/kaldi_data/singleword/en/test /de/ /fr/ /es/ 
# Multilingual Librispeech: /training-1/asr/kaldi_data/mls/en/test /de/ /fr/ /es/
# Voxpopuli: /training-1/voxpopuli/dataset_with_spaces_voxpopuli_en/test *_de *_fr *_es
datasets = {}
for dataset_path, dataset_title in dataset_paths:
    dataset = []
    wav_scp_dict = {}
    with open(f"{dataset_path}/wav.scp", encoding="UTF-8") as rf:
        for line in rf:
            id, wav_path = line.strip().split("\t")
            wav_scp_dict[id] = wav_path
    
    text_dict = {}
    with open(f"{dataset_path}/text", encoding="UTF-8") as rf:
        for line in rf:
            id, text = line.strip().split("\t")
            text_dict[id] = text
    
    for id, wav_path in wav_scp_dict.items():
        text = text_dict[id]
        dataset.append({"audio" : wav_path, "text" : text})
    datasets[dataset_title] = dataset

#helper function: load audio and make sure it is at 16kHz
def load_audio(audio_path):
    waveform, sample_rate = torchaudio.load(audio_path)

    waveform = torchaudio.functional.resample(
        waveform, sample_rate, 16000
    )

    waveform = waveform.mean(dim=0)
    return waveform
'''
first_entry = dataset[0]
print(first_entry)
print("-"*10)
print(first_entry["text"])
audio = load_audio(first_entry["audio"])
display(Audio(audio, rate=16000))
'''


'\nfirst_entry = dataset[0]\nprint(first_entry)\nprint("-"*10)\nprint(first_entry["text"])\naudio = load_audio(first_entry["audio"])\ndisplay(Audio(audio, rate=16000))\n'

## Transcription Function

In [4]:
def transcribe(audio_path, prompt=None, language=None):
    kwargs = {"audio": audio_path}
    if prompt:
        kwargs["prompt"] = prompt
    if language:
        kwargs["language"] = language
    inputs = processor.apply_transcription_request(**kwargs).to(model.device, model.dtype)

    output_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    return processor.decode(generated_ids, return_format="transcription_only")[0].strip()

## Text Normalization, Imports

In [5]:
import jiwer
import re
import string

def normalize(text):
    text = text.translate(str.maketrans('', '', string.punctuation)).lower().strip()
    text = re.sub(r" +", " ", text).strip()
    return text

## Run Inference on the Test Set

In [9]:
from tqdm.auto import tqdm
import pandas as pd
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_trf")
context_list = [False, True]

for c in context_list:
    for dataset_title, dataset_entries in datasets.items():
        eval_rows = []
        for sample in tqdm(dataset_entries[:50]):
            entities = None
            if c:
                doc = nlp(sample["text"])
                entities = [ent.text for ent in doc.ents]
                entities = ", ".join(entities)
            model_output = None
            if entities:
                model_output = transcribe(sample["audio"], prompt=entities)
            else:
                model_output = transcribe(sample["audio"])
            wer = jiwer.wer(sample["text"], model_output)
            cer = jiwer.cer(sample["text"], model_output)
            eval_rows.append({"audio_path": sample["audio"],
                              "label": normalize(sample["text"]),
                              "prediction": normalize(model_output),
                              "wer" : wer,
                              "cer": cer
                             }
                         )
        output_statistics = []
        results = pd.DataFrame(eval_rows)
        results.head()
        output_statistics.append("Mean WER:", results["wer"].mean())
        output_statistics.append("Mean CER:", results["cer"].mean())
        results = results.sort_values("wer", ascending=False)
        output_name = ""
        if c:
            output_name = f"qwen_{dataset_title}_with_context"
        else:
            output_name = f"qwen_{dataset_title}_without_context"
        results.to_csv(f"{output_name}.tsv", sep="\t", index=False)
        with open(f"{output_name}_statistics.txt", "w", encoding="UTF-8") as wf:
            for line in output_statistics:
                wf.write(line+"\n")
        print("="*50)
        print(dataset_title)
        print(output_name)
        print("\n".join(output_statistics))
        print("="*50)

  0%|          | 0/50 [00:00<?, ?it/s]

NameError: name 'dataset_name' is not defined

## Corpus-Level Results

In [17]:
print("Mean WER:", results["wer"].mean())
print("Mean CER:", results["cer"].mean())
results[["wer", "cer"]].describe()
results = results.sort_values("wer", ascending=False)
results.to_csv("results_with_context.tsv", sep="\t", index=False)

Mean WER: 0.06017530966813147
Mean CER: 0.032810756318717306


## Worst Samples

In [11]:
results.sort_values("wer", ascending=False).head(10)


,audio_path,label,prediction,wer,cer
35,/training-1/asr/voxpopuli/out/transcribed_data...,it is unacceptable and as has been said a sign...,i think it is unacceptable and it has been sai...,0.250000,0.127907
54,/training-1/asr/voxpopuli/out/transcribed_data...,is there contact with the syrian national coun...,are there contacts with the syrian national co...,0.250000,0.103448
78,/training-1/asr/voxpopuli/out/transcribed_data...,should these be offensive then they can be tac...,and should these be offensive then they can be...,0.230769,0.095890
96,/training-1/asr/voxpopuli/out/transcribed_data...,so with todays resolution we aim to fulfil our...,so with todays resolution we aim to fulfil our...,0.210526,0.062893
94,/training-1/asr/voxpopuli/out/transcribed_data...,i really hope that through your vote tomorrow ...,i really hope through your vote that you will ...,0.173913,0.151261
19,/training-1/asr/voxpopuli/out/transcribed_data...,the situation in which social and economic ris...,the situation in which the social and economic...,0.170213,0.066667
53,/training-1/asr/voxpopuli/out/transcribed_data...,i would like to hear from mrs ashton what kind...,i would like to hear from lady ashton what kin...,0.166667,0.083333
17,/training-1/asr/voxpopuli/out/transcribed_data...,however one aspect was left out the situation ...,one aspect was left aside the situation of tho...,0.162791,0.150198
0,/training-1/asr/voxpopuli/out/transcribed_data...,then we as a parliament could take our respons...,we as a parliament could take our responsibili...,0.160000,0.143836
2,/training-1/asr/voxpopuli/out/transcribed_data...,we need to take a quick decision if we want to...,we need to take a quick decision if we want to...,0.156250,0.076923


In [12]:
from IPython.display import Audio, display

for _, row in results.sort_values("wer", ascending=False).head(10).iterrows():
    print("=" * 70)
    print("AUDIO  :", row["audio_path"])
    print("LABEL    :", row["label"])
    print("PRED    :", row["prediction"])
    print(f"WER {row["wer"]:.2%} | CER {row["cer"]:.2%}")
    display(Audio(load_audio(row["audio_path"]), rate=16000))


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2018/20180312-0900-PLENARY-14-en_20180312-20:14:51_4.wav
LABEL    : it is unacceptable and as has been said a sign of general regression for womens rights
PRED    : i think it is unacceptable and it has been said a sign of general regression for womens rights
WER 25.00% | CER 12.79%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2012/20120612-0900-PLENARY-11-en_20120612-17:59:38_5.wav
LABEL    : is there contact with the syrian national council and if so could she elaborate on this
PRED    : are there contacts with the syrian national council and if so could you elaborate on these
WER 25.00% | CER 10.34%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2017/20170405-0900-PLENARY-13-en_20170405-15:58:20_4.wav
LABEL    : should these be offensive then they can be tackled by a forthright debate
PRED    : and should these be offensive then they can be tackled via forthright debate
WER 23.08% | CER 9.59%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2018/20180614-0900-PLENARY-4-en_20180614-10:28:47_3.wav
LABEL    : so with todays resolution we aim to fulfil our moral and political duty as european democrats by trying to save mr sentsovs life and help regain his freedom as well as that of all the other names in the resolution held in russian captivity deprived of freedom and human dignity and even of legal and medical assistance
PRED    : so with todays resolution we aim to fulfil our moral and political duty as european democrats by trying to save mr sentsovs lives and help regain his freedom as well as those of all other names in the resolution held in russian captivity deprived of freedom and human dignity even of legal and medical assistance
WER 21.05% | CER 6.29%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131023-0900-PLENARY-14-en_20131023-19:30:09_7.wav
LABEL    : i really hope that through your vote tomorrow you will give us your support so as not to lose any of these achievements
PRED    : i really hope through your vote that you will give us your support not to lose any of these achievements
WER 17.39% | CER 15.13%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2009/20090310-0900-PLENARY-20-en_20090310-22:11:44_2.wav
LABEL    : the situation in which social and economic risks are made worse by the absence of parents who cannot meet their childrens needs for care protection and education may lead to an increase in vulnerability the primary responsibility for the development of the child rests with the parents
PRED    : the situation in which the social and economic risks are made worse by the absence of parents who cannot meet their childrens need for care protection and education may bring about increased vulnerability the primary responsibility for the development of the child rests with the parents
WER 17.02% | CER 6.67%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2012/20120612-0900-PLENARY-11-en_20120612-17:59:38_4.wav
LABEL    : i would like to hear from mrs ashton what kind of contact there is with syrian opposition groups
PRED    : i would like to hear from lady ashton what kind of contacts there are with syrian opposition groups
WER 16.67% | CER 8.33%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2009/20090310-0900-PLENARY-20-en_20090310-22:15:43_4.wav
LABEL    : however one aspect was left out the situation of thousands of european children left behind by parents who go to work in another european country generally called the migration orphans of which there are almost three hundred and fifty zero in my country
PRED    : one aspect was left aside the situation of thousands of european children left behind by their parents who go to work in another european country generally called the migration orphans almost three hundred and fifty zero in my country
WER 16.28% | CER 15.02%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131022-0900-PLENARY-3-en_20131022-08:32:08_10.wav
LABEL    : then we as a parliament could take our responsibility and quickly vote through these measures because they will need to take effect on one january
PRED    : we as a parliament could take our responsibility and quickly vote through these measures because they need to take effect actually only one january
WER 16.00% | CER 14.38%


AUDIO  : /training-1/asr/voxpopuli/out/transcribed_data/en/2013/20131022-0900-PLENARY-3-en_20131022-08:32:08_9.wav
LABEL    : we need to take a quick decision if we want to follow the normal procedure i would urge the council to withdraw its legal basis and to accept the commissions legal basis
PRED    : we need to take a quick decision if we want to follow the normal procedure i would urge the council to withdraw their legal base and accept the commission legal base
WER 15.62% | CER 7.69%
